# Football detection dashboard

**Dashboard only. No training at startup. No dataset or API key required.**

Run it from the repo root with `uv run python main.py app`, then open http://127.0.0.1:7860. It loads the committed `weights/best.pt` and the files `train.ipynb` exported to `outputs/`: test-set ground truth and predictions, per-class metrics, cost assumptions, the cost sweep and the validation-selected thresholds. It never calls training or reruns dataset evaluation. Uploaded-image inference and saved labelled evaluation are separate: an unlabelled uploaded image does not change the evaluation metrics.

**Colab alternative:** open this notebook in a fresh Colab session and choose **Runtime → Run all**. If `weights/` and `outputs/` are not beside the notebook, it asks you to upload `assignment2_results.zip` (exported by `train.ipynb`). In Colab the last cell creates a temporary public `gradio.live` link, and the session must keep running. GPU is optional: saved metrics and sliders use the CPU; a GPU only speeds up inference on uploaded images.

## 1. Dependencies (Colab only)

Under `uv run`, everything is already installed from `uv.lock`, so this cell does nothing. In a fresh Colab session it installs the dashboard's dependencies; it does not train a model.

In [ ]:
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules
MODEL_PACKAGE_VERSION = "8.4.191"  # Ultralytics version that saved weights/best.pt (outputs/environment.json).

if IN_COLAB:
    loaded = sys.modules.get("ultralytics")
    loaded_version = getattr(loaded, "__version__", None)
    if loaded_version and loaded_version != MODEL_PACKAGE_VERSION:
        raise RuntimeError(
            f"Ultralytics {loaded_version} is already imported. Run "
            f"%pip install -q ultralytics=={MODEL_PACKAGE_VERSION}, then Runtime → Restart session "
            "and rerun this dashboard notebook."
        )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         f"ultralytics=={MODEL_PACKAGE_VERSION}", "gradio>=5,<7", "pandas", "matplotlib", "pillow"],
        check=True,
    )
    print("Dashboard dependencies ready.")
else:
    print("Using the installed environment; nothing to install.")

## 2. Load the saved weights and pipeline outputs

In the repo these are `weights/best.pt` and `outputs/*.json`. If any are missing, the cell restores just those files from `assignment2_results.zip` (asking for an upload in Colab). The ZIP itself is not modified.

In [ ]:
from pathlib import Path
import json
import shutil
from zipfile import ZipFile

import numpy as np
import torch

ROOT = Path.cwd().resolve()
WEIGHTS = ROOT / "weights"
OUT = ROOT / "outputs"
ZIP_PATH = ROOT / "assignment2_results.zip"  # Only needed when weights/ or outputs/ are missing (e.g. Colab).
REQUIRED_FILES = ["weights/best.pt", "outputs/test_predictions.json", "outputs/comparison.json",
                  "outputs/costing.json", "outputs/training_settings.json"]


def missing_files():
    return [name for name in REQUIRED_FILES if not (ROOT / name).is_file()]


if missing_files():
    if not ZIP_PATH.is_file() and IN_COLAB:
        from google.colab import files
        print("Choose your saved assignment2_results.zip:")
        uploaded = files.upload()
        choices = [Path(name) for name in uploaded if name.lower().endswith(".zip")]
        if len(choices) != 1:
            raise RuntimeError("Upload exactly one results ZIP, then rerun this cell.")
        ZIP_PATH = ROOT / choices[0]
        del uploaded
    if ZIP_PATH.is_file():
        # Copy out only the files the dashboard reads; no dataset, API key or training needed.
        with ZipFile(ZIP_PATH) as archive:
            absent = sorted(set(missing_files()) - set(archive.namelist()))
            if absent:
                raise RuntimeError("ZIP is missing required saved files: " + ", ".join(absent))
            for name in missing_files():
                target = ROOT / name
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(name) as source, target.open("wb") as handle:
                    shutil.copyfileobj(source, handle)
    if missing_files():
        raise FileNotFoundError(
            "Missing " + ", ".join(missing_files()) + ". Run `uv run python main.py train` first, "
            "or put assignment2_results.zip beside this notebook."
        )


def read_json(name):
    return json.loads((OUT / name).read_text(encoding="utf-8"))


prediction_export = read_json("test_predictions.json")
comparison_export = read_json("comparison.json")
costing_export = read_json("costing.json")
training_export = read_json("training_settings.json")

CLASS_NAMES = list(prediction_export["class_names"])
GT = {}
P_FT = {}
for name, data in prediction_export["images"].items():
    GT[name] = np.asarray(data["gt"], dtype=float).reshape(-1, 5)
    P_FT[name] = np.asarray(data["pred"], dtype=float).reshape(-1, 6)
    for array in (GT[name], P_FT[name]):
        if not np.isfinite(array).all():
            raise ValueError(f"Non-finite values in saved predictions for {name}.")
        if len(array) and not np.isin(array[:, 0], np.arange(len(CLASS_NAMES))).all():
            raise ValueError(f"Invalid class id for {name}.")
if not GT:
    raise RuntimeError("outputs/test_predictions.json contains no labelled test predictions.")
if len(GT) != comparison_export["n_test_images"]:
    raise RuntimeError("Prediction count differs from the saved comparison metadata.")

results = comparison_export["results"]
POS_CLASS = costing_export["positive_class"]
costs = costing_export["costs"]
C_FN, C_FP = float(costs["C_FN"]), float(costs["C_FP"])
C_TP, C_REVIEW = float(costs["C_TP"]), float(costs["C_REVIEW"])
T_STAR = float(costing_export["t_star"])
T_LOW, T_HIGH = float(costing_export["t_low"]), float(costing_export["t_high"])
if not 0.001 <= T_LOW <= T_HIGH <= 1 or not 0.001 <= T_STAR <= 1:
    raise ValueError("Saved thresholds are outside the supported prediction confidence range.")
volume = costing_export["volume"]
MATCHES_PER_DAY = float(volume["matches_per_day"])
INCIDENTS_PER_MATCH = float(volume["candidate_frames_per_match"])
SECONDS_PER_REVIEW = float(volume["seconds_per_review"])
REVIEWER_HOURS_PER_DAY = float(volume["productive_hours_per_reviewer"])
saved_sweep = costing_export["sweep"]
thrs = np.asarray(saved_sweep["thresholds"], dtype=float)
valid_sweep = saved_sweep.get("validation_cost_per_1000")  # The curve t* was chosen on.
valid_sweep = None if valid_sweep is None else np.asarray(valid_sweep, dtype=float)
sweep = np.column_stack([
    saved_sweep["test_cost_per_1000"], np.zeros(len(thrs)),
    saved_sweep["test_FP"], saved_sweep["test_FN"],
])
single = costing_export["test_model_alone_at_validation_threshold"]
TEST_SINGLE = (single["cost_per_1000"], single["TP"], single["FP"], single["FN"])
IMGSZ = training_export.get("imgsz", 960)
DEVICE = 0 if torch.cuda.is_available() else "cpu"
selection = training_export.get("selected_weights_kind", "saved export")
model_description = ("final-epoch export (historical best was unavailable)"
                     if selection == "final_checkpoint_fallback" else "saved selected checkpoint")

print(f"Loaded {len(GT)} labelled test images; classes: {CLASS_NAMES}.")
print(f"Saved policy: t*={T_STAR:.2f}; review band [{T_LOW:.2f}, {T_HIGH:.2f}).")
print(f"Model: {model_description}; device: {DEVICE}.")
print("Next: build and launch the dashboard. No training has run.")

def iou(a, b):
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2-x1, 0, None) * np.clip(y2-y1, 0, None)
    aa = (a[:, 2]-a[:, 0])*(a[:, 3]-a[:, 1]); ab = (b[:, 2]-b[:, 0])*(b[:, 3]-b[:, 1])
    return inter / (aa[:, None] + ab[None, :] - inter + 1e-9)

def match_image(pred, gt, cls, iou_thr=0.5):
    # greedy by confidence: returns (conf of preds sorted desc, is_tp flags, matched_gt_idx, n_gt)
    P = pred[pred[:, 0] == cls]; G = gt[gt[:, 0] == cls]
    P = P[np.argsort(-P[:, 1], kind="stable")]
    tp = np.zeros(len(P), bool); mg = -np.ones(len(P), int)
    if len(P) and len(G):
        M = iou(P[:, 2:6], G[:, 1:5]); used = np.zeros(len(G), bool)
        for i in range(len(P)):
            cand = np.where((M[i] >= iou_thr) & ~used)[0]
            if len(cand):
                j = cand[np.argmax(M[i, cand])]; used[j] = True; tp[i] = True; mg[i] = j
    return P[:, 1], tp, mg, len(G)

## 3. Open the dashboard

- Upload a football image: green player boxes pass, orange candidates go to review, and candidates below the lower threshold fail and are hidden. Football boxes are shown above the upper threshold.
- Move both thresholds: the player TP/FP/FN matrix, scenario cost per 1,000 frames, review share, per-class precision/recall and policy comparison update. TN is undefined for object detection and is shown as an unavailable cell.
- Inspect the cost curve: the validation sweep that chose the threshold and the held-out test sweep are both drawn, the saved validation-selected threshold is marked; both current slider positions are also marked. The saved chosen policy is not retuned on test data.
- Per-class AP50/mAP50–95 are saved ranking metrics and stay fixed; precision/recall and error counts respond to thresholds. AP is not defined here for the simulated review policy.

Human review assumes perfect confirmation/rejection of candidates in the review band, while unproposed players remain missed. Costs are illustrative assumptions. Pass/fail describes detection handling, never an offside judgement.

**Saved model provenance:** `outputs/training_settings.json` records a final-checkpoint fallback. `weights/best.pt` is the final (epoch-40) model, because the historical best checkpoint from before the Colab interruption was unavailable. The app states this at the top.

In [ ]:
# In Colab this creates the temporary link needed to open the app. Locally the app is
# served on http://127.0.0.1:7860 (set GRADIO_SERVER_PORT to change the port).
SHARE_DEMO = IN_COLAB

import math
from functools import lru_cache

import gradio as gr
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # Figures are rendered off-screen and sent to Gradio; safe in its worker threads.
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from ultralytics import YOLO

# ---------- Check the ZIP restore cell has finished ----------
required = [
    "WEIGHTS", "CLASS_NAMES", "P_FT", "GT", "match_image",
    "POS_CLASS", "C_FP", "C_FN", "C_TP", "C_REVIEW",
    "T_STAR", "T_LOW", "T_HIGH", "thrs", "sweep", "results", "TEST_SINGLE",
    "MATCHES_PER_DAY", "INCIDENTS_PER_MATCH",
    "SECONDS_PER_REVIEW", "REVIEWER_HOURS_PER_DAY",
    "IMGSZ", "DEVICE"
]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(
        "Run the ZIP restore cell first. Missing: " + ", ".join(missing)
    )

weights_path = WEIGHTS / "best.pt"
if not weights_path.exists():
    raise FileNotFoundError(f"Trained weights not found: {weights_path}")

dashboard_model = YOLO(str(weights_path))  # Loads weights only. No training.
model_names = dashboard_model.names
model_names = ([model_names[k] for k in sorted(model_names, key=int)]
               if isinstance(model_names, dict) else list(model_names))
if model_names != CLASS_NAMES:
    raise RuntimeError("The model's class order differs from the exported evaluation classes.")
positive_id = CLASS_NAMES.index(POS_CLASS)
n_frames = len(GT)
if not n_frames:
    raise RuntimeError("No held-out predictions available.")

daily_frames = MATCHES_PER_DAY * INCIDENTS_PER_MATCH

# Match once at IoU >= 0.50, using the notebook's existing matcher.
# Lower-confidence predictions cannot change earlier, higher-confidence matches.
cached_matches = {
    c: [match_image(P_FT[k], GT[k], c, 0.5) for k in GT]
    for c in range(len(CLASS_NAMES))
}

positive_frames = sum(
    bool(np.any(g[:, 0] == positive_id)) for g in GT.values()
)
positive_objects = sum(
    int(np.sum(g[:, 0] == positive_id)) for g in GT.values()
)

# ---------- Metrics from cached predictions ----------
@lru_cache(maxsize=512)
def detection_counts(class_id, threshold):
    tp = fp = fn = 0
    for conf, matched, _, n_gt in cached_matches[class_id]:
        keep = conf >= threshold
        found = int(matched[keep].sum())
        tp += found
        fp += int((~matched[keep]).sum())
        fn += n_gt - found
    return tp, fp, fn


@lru_cache(maxsize=1024)
def review_counts(low, high):
    """Same conservative review assumption as section 10."""
    tp = fp = fn = reviewed = 0

    for conf, matched, _, n_gt in cached_matches[positive_id]:
        keep = conf >= low
        band = keep & (conf < high)

        reviewed += int(band.any())
        found = int(matched[keep].sum())
        tp += found
        # Reviewer rejects false positives within the review band.
        fp += int((~matched & keep & ~band).sum())
        # Reviewer does not recover players missed below the lower threshold.
        fn += n_gt - found

    cost = (
        fp * C_FP + fn * C_FN + tp * C_TP + reviewed * C_REVIEW
    ) / n_frames * 1000

    return tp, fp, fn, reviewed, cost


def metric_row(name, policy, counts):
    tp, fp, fn = counts
    ap = results["finetuned"].get(name, {})
    return {
        "Class": name,
        "Policy": policy,
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "Precision": tp / (tp + fp) if tp + fp else np.nan,
        "Recall": tp / (tp + fn) if tp + fn else np.nan,
        "AP50*": np.nan if policy == "After simulated review" else ap.get("AP50", np.nan),
        "mAP50-95*": np.nan if policy == "After simulated review" else ap.get("mAP50-95", np.nan),
    }


def policy_row(name, cost, review_share):
    reviews = daily_frames * review_share
    hours = reviews * SECONDS_PER_REVIEW / 3600
    fte = hours / REVIEWER_HOURS_PER_DAY
    return {
        "Strategy": name,
        "€/1,000 frames": cost,
        "€/day": cost * daily_frames / 1000,
        "Review %": review_share * 100,
        "Reviews/day": reviews,
        "Review hours/day": hours,
        "Reviewer FTE": fte,
        "People needed**": math.ceil(fte),
    }


# ---------- Upload inference, cached so sliders do not rerun YOLO ----------
@lru_cache(maxsize=8)
def predict_upload(path):
    return dashboard_model.predict(
        source=path,
        conf=0.001,
        iou=0.6,
        imgsz=IMGSZ,
        device=DEVICE,
        max_det=300,
        verbose=False,
    )[0]


def draw_upload(path, low, high):
    if path is None:
        return None, "Upload an image to inspect detections."

    prediction = predict_upload(path)
    image = Image.fromarray(prediction.orig_img[:, :, ::-1]).copy()
    draw = ImageDraw.Draw(image)
    width = max(2, image.width // 400)

    player_scores = []

    for box in prediction.boxes:
        cls = int(box.cls.item())
        score = float(box.conf.item())

        if cls == positive_id:
            player_scores.append(score)
            if score < low:
                continue
            state = "REVIEW" if score < high else "PASS"
            colour = "orange" if state == "REVIEW" else "lime"
        else:
            if score < high:
                continue
            state, colour = "DETECTED", "cyan"

        xyxy = box.xyxy[0].cpu().tolist()
        draw.rectangle(xyxy, outline=colour, width=width)
        draw.text(
            (xyxy[0], max(0, xyxy[1] - 12)),
            f"{CLASS_NAMES[cls]} {score:.2f} | {state}",
            fill=colour,
            stroke_width=1,
            stroke_fill="black",
        )

    if any(low <= score < high for score in player_scores):
        status = "HUMAN REVIEW — at least one player is in the review band."
    elif any(score >= high for score in player_scores):
        status = "PASS — detections accepted; other players may still be missed. No offside decision."
    else:
        status = "FAIL — no player detection accepted. Inspect manually; no offside decision."

    return image, status


# ---------- Dashboard callback ----------
def refresh_dashboard(low, high, uploaded_path):
    low, high = round(float(low), 3), round(float(high), 3)
    if low > high:
        raise gr.Error("The lower threshold must be ≤ the upper threshold.")

    tp, fp, fn, reviewed, routed_cost = review_counts(low, high)
    share = reviewed / n_frames

    auto_tp, auto_fp, auto_fn = detection_counts(positive_id, high)
    auto_cost = (
        auto_fp * C_FP + auto_fn * C_FN + auto_tp * C_TP
    ) / n_frames * 1000

    summary = (
        f"### Scenario €{routed_cost:,.2f} per 1,000 test frames\n"
        f"**Sent for review:** {share:.1%} · "
        f"**Reviews/day:** {daily_frames * share:.1f} · "
        f"**Review time/day:** "
        f"{daily_frames * share * SECONDS_PER_REVIEW / 3600:.2f} hours\n\n"
        f"Player detections: **discard < {low:.2f}**, "
        f"**review [{low:.2f}, {high:.2f})**, "
        f"**accept ≥ {high:.2f}**."
    )

    # Detection has no finite set of true-negative background objects.
    fig_cm, ax = plt.subplots(figsize=(5, 3.8))
    matrix = np.ma.array([[tp, fp], [fn, 0]], mask=[[False, False], [False, True]])
    colours = plt.get_cmap("Blues").copy()
    colours.set_bad("#eeeeee")
    ax.imshow(matrix, cmap=colours)
    for row, col, label in [
        (0, 0, f"TP\n{tp:,}"),
        (0, 1, f"FP\n{fp:,}"),
        (1, 0, f"FN\n{fn:,}"),
        (1, 1, "TN\nnot defined"),
    ]:
        ax.text(col, row, label, ha="center", va="center")
    ax.set_xticks([0, 1], ["Labelled player", "No matched player"])
    ax.set_yticks([0, 1], ["Accepted player", "No accepted match"])
    ax.set_xlabel("Ground truth / matching outcome")
    ax.set_ylabel("Final detection outcome")
    ax.set_title("Player error counts after simulated review")
    fig_cm.tight_layout()
    plt.close(fig_cm)

    fig_cost, ax = plt.subplots(figsize=(7, 3.8))
    if valid_sweep is not None:
        ax.plot(thrs, valid_sweep, color="grey", alpha=0.8,
                label="Validation: curve used to pick t*")
    ax.plot(thrs, sweep[:, 0], label="Test: exploratory curve")
    ax.axvline(
        T_STAR, color="green", linestyle="--",
        label=f"Validation-selected t*: {T_STAR:.2f}"
    )
    ax.axvline(low, color="orange", linestyle=":", label=f"Lower: {low:.2f}")
    ax.axvline(high, color="red", linestyle=":", label=f"Upper: {high:.2f}")
    ax.scatter(
        [high], [auto_cost], color="red", zorder=5,
        label="Current model-only cost"
    )
    ax.set(
        xlabel="Confidence threshold",
        ylabel="€ per 1,000 frames",
        title="Illustrative test cost — exploratory sliders",
    )
    ax.legend(fontsize=8)
    fig_cost.tight_layout()
    plt.close(fig_cost)

    rows = [
        metric_row(name, f"Model alone, threshold {high:.2f}",
                   detection_counts(c, high))
        for c, name in enumerate(CLASS_NAMES)
    ]
    rows.append(metric_row(POS_CLASS, "After simulated review", (tp, fp, fn)))
    per_class_table = pd.DataFrame(rows).round(4)

    comparison = pd.DataFrame([
        policy_row(
            "Predict nothing proxy",
            positive_objects * C_FN / n_frames * 1000, 0
        ),
        policy_row("Person checks every frame†", 1000 * C_REVIEW, 1),
        policy_row("Model alone — current upper threshold", auto_cost, 0),
        policy_row("Model alone — validation-selected t*", float(TEST_SINGLE[0]), 0),
        policy_row("Model + current review band", routed_cost, share),
    ]).round(3)

    preview, status = draw_upload(uploaded_path, low, high)
    return summary, fig_cm, fig_cost, per_class_table, comparison, preview, status


# ---------- Interface ----------
# Close only a previous instance of this dashboard when rerunning the cell.
if "football_dashboard" in globals():
    football_dashboard.close()

with gr.Blocks(title="Football Detection — Cost & Human Review") as football_dashboard:
    gr.Markdown(
        f"**Saved model:** {model_description}. **Evaluated images:** {len(GT)}. "
        "Startup loads saved weights and JSON only. No training or dataset download is performed."
    )
    gr.Markdown(
        "# Football detection: cost and human review\n"
        "**Early-stage demo: a first step towards accessible amateur-football analysis.** "
        "Detects players and balls; does not decide offside, identify teams or draw a calibrated line. "
        "Accept/discard describes a candidate detection only. High confidence does not prove a frame is complete. "
        "Review is simulated, and euro values are assumptions. Initial thresholds were chosen on validation; "
        "test-set slider exploration does not change the exported fixed-policy report."
    )

    with gr.Row():
        low_slider = gr.Slider(
            minimum=0.001, maximum=1.0, value=float(T_LOW),
            step=0.001, label="Lower threshold: fail/discard → human review"
        )
        high_slider = gr.Slider(
            minimum=0.001, maximum=1.0, value=float(T_HIGH),
            step=0.001, label="Upper threshold: human review → pass/accept"
        )

    summary_box = gr.Markdown()

    with gr.Tab("Costs and evaluation"):
        with gr.Row():
            confusion_plot = gr.Plot(label="Positive-class error matrix")
            cost_plot = gr.Plot(label="Cost against threshold")

        class_table = gr.Dataframe(label="Per-class results", interactive=False)
        strategy_table = gr.Dataframe(
            label="Strategy costs and staffing", interactive=False
        )

        gr.Markdown(
            f"**Evaluation population:** {n_frames} held-out frames; "
            f"{positive_frames / n_frames:.1%} contain a labelled player; "
            f"{positive_objects / n_frames:.2f} labelled players per frame. "
            "Scenario projections assume future candidate frames have this same mix; this is unverified.\n\n"
            f"**Assumptions:** FP €{C_FP:g}/player; FN €{C_FN:g}/player; "
            f"TP €{C_TP:g}/player; review €{C_REVIEW:g}/frame; "
            f"{SECONDS_PER_REVIEW:g} seconds/review; "
            f"{daily_frames:g} frames/day; "
            f"{REVIEWER_HOURS_PER_DAY:g} productive hours/person/day.\n\n"
            "**Review simulation:** a frame is reviewed if any player detection "
            "falls in the band. The reviewer correctly confirms/rejects those "
            "candidates; players missed below the lower threshold remain missed. "
            "Only player errors are monetised.\n\n"
            "*AP figures are saved model ranking metrics (blank for simulated review); "
            "precision, recall and error counts change with the sliders. "
            "**People needed assumes work can be pooled across the day. "
            "†The check-every-frame comparison assumes a perfect human reviewer.\n\n"
            "**Metrics use the labelled evaluation set. An uploaded image has "
            "no ground-truth labels and does not change evaluation results.**"
        )

    with gr.Tab("Upload an image"):
        with gr.Row():
            upload = gr.Image(type="filepath", label="Upload football image")
            annotated = gr.Image(label="Predictions", interactive=False)
        image_status = gr.Markdown()
        gr.Markdown(
            "Green = accepted player; orange = player requiring review; "
            "cyan = another class above the upper threshold. "
            "Discarded boxes are hidden."
        )

    inputs = [low_slider, high_slider, upload]
    outputs = [
        summary_box, confusion_plot, cost_plot,
        class_table, strategy_table, annotated, image_status
    ]

    football_dashboard.load(refresh_dashboard, inputs, outputs)
    low_slider.change(refresh_dashboard, inputs, outputs)
    high_slider.change(refresh_dashboard, inputs, outputs)
    upload.change(refresh_dashboard, inputs, outputs)

# As a script (main.py app) launch() blocks and keeps serving; in a notebook it returns.
football_dashboard.queue().launch(share=SHARE_DEMO, debug=False)